# Bloque 3 — Agentes de IA
## Notebook 01: Equipo de tres agentes investigando ContiLeaks

---

### Objetivo

Construir una **crew de tres agentes especializados** que analicen los datos ya procesados de ContiLeaks y produzcan un informe de inteligencia táctica.

Cada agente lee datos distintos y pasa su output al siguiente:

```
┌──────────────────┐     ┌─────────────────────┐     ┌──────────────────┐
│   INVESTIGADOR   │────▶│      ANALISTA        │────▶│    REDACTOR      │
│                  │     │                      │     │                  │
│ Lee actor_       │     │ Lee conti_sample_    │     │ Recibe outputs   │
│ profiles.json    │     │ classified.parquet   │     │ de los otros dos │
│                  │     │                      │     │                  │
│ Output:          │     │ Output:              │     │ Output:          │
│ Top 5 actores    │     │ Patrones por actor   │     │ Informe TI final │
└──────────────────┘     └─────────────────────┘     └──────────────────┘
```

### Datos necesarios (ya en `data_para_alumnos/`)
- `ContiLeaks/data/processed/actor_profiles.json` (20 KB)
- `ContiLeaks/data/processed/conti_sample_classified.parquet` (88 KB)

In [1]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os
import json
import concurrent.futures
from pathlib import Path

import pandas as pd

os.environ['OPENAI_API_KEY'] = 'NA'

from crewai import Agent, Task, Crew, Process, LLM


def kickoff(crew, timeout=600):
    """Ejecuta crew.kickoff() en un hilo separado para evitar conflictos
    con el event loop de Jupyter/nbconvert (problema de CrewAI 1.x)."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff).result(timeout=timeout)


# ─── RUTAS ───────────────────────────────────────────────────────────────────
DATA = Path('..') / 'data_para_alumnos' / 'ContiLeaks' / 'data' / 'processed'

PROFILES_JSON   = DATA / 'actor_profiles.json'
CLASSIFIED_PARQ = DATA / 'conti_sample_classified.parquet'

assert PROFILES_JSON.exists(),   f'No encontrado: {PROFILES_JSON}'
assert CLASSIFIED_PARQ.exists(), f'No encontrado: {CLASSIFIED_PARQ}'

print('Ficheros de datos localizados correctamente.')

Ficheros de datos localizados correctamente.


In [2]:
# ─── CARGAR DATOS ────────────────────────────────────────────────────────────

# Cargamos los perfiles de actores. Son 30 actores con rol, confianza,
# resumen y ejemplos de mensajes inferidos por el LLM en bloque4.
with open(PROFILES_JSON) as f:
    profiles = json.load(f)

# Cargamos el DataFrame con los mensajes clasificados.
# 1500 mensajes de los 30 actores más activos, con categoría asignada.
df = pd.read_parquet(CLASSIFIED_PARQ)

print(f'Actores con perfil : {len(profiles)}')
print(f'Mensajes clasificados: {len(df)}')
print(f'Categorías disponibles: {sorted(df["category"].unique())}')

Actores con perfil : 30
Mensajes clasificados: 1500
Categorías disponibles: ['comms', 'financial', 'operational', 'organizational', 'technical', 'unknown']


In [3]:
# ─── PREPARAR DATOS COMO TEXTO ───────────────────────────────────────────────
# Los agentes de CrewAI se comunican exclusivamente mediante texto.
# Necesitamos convertir nuestros DataFrames y dicts en cadenas de texto
# que podamos incluir en las descripciones de las tareas.

# TEXTO 1: Resumen de perfiles para el agente Investigador
# Convertimos el JSON de perfiles en un texto estructurado y legible.
lineas_perfiles = []
for actor, info in profiles.items():
    linea = (
        f"- {actor}: rol={info['role']}, confianza={info['confidence']}. "
        f"{info['summary'][:150]}..."
    )
    lineas_perfiles.append(linea)

TEXTO_PERFILES = 'PERFILES DE ACTORES DE CONTI:\n' + '\n'.join(lineas_perfiles)

# TEXTO 2: Distribución de categorías por actor para el agente Analista
# Calculamos cuántos mensajes tiene cada actor en cada categoría.
cats_por_actor = (
    df.groupby(['username', 'category'])
    .size()
    .unstack(fill_value=0)
)

lineas_cats = []
for actor in cats_por_actor.index:
    row = cats_por_actor.loc[actor]
    total = row.sum()
    dominante = row.idxmax()
    lineas_cats.append(
        f"- {actor}: {total} msgs, dominante={dominante} "
        f"({', '.join(f'{c}={v}' for c,v in row.items() if v > 0)})"
    )

TEXTO_CATEGORIAS = 'DISTRIBUCIÓN DE CATEGORÍAS POR ACTOR:\n' + '\n'.join(lineas_cats)

print('Textos de contexto preparados.')
print(f'Perfiles: {len(TEXTO_PERFILES)} caracteres')
print(f'Categorías: {len(TEXTO_CATEGORIAS)} caracteres')

Textos de contexto preparados.
Perfiles: 5750 caracteres
Categorías: 3163 caracteres


## Definir los tres agentes

Cada agente tiene una especialización distinta. El `backstory` es importante: le dice al modelo cómo debe comportarse, qué tono usar y qué limitaciones respetar.

In [4]:
# ─── CONFIGURACIÓN DEL LLM ───────────────────────────────────────────────────
# Los tres agentes comparten el mismo modelo base (qwen2.5:14b en Ollama).
# En un sistema real podrías usar modelos diferentes para tareas distintas.
llm = LLM(
    model='ollama/qwen2.5:14b',
    base_url='http://localhost:11434',
)

# ─── AGENTE 1: INVESTIGADOR ───────────────────────────────────────────────────
# Su trabajo es leer los perfiles de actores e identificar los más relevantes.
# El backstory lo posiciona como alguien que ya conoce los datos y sabe qué buscar.
investigador = Agent(
    role='Investigador de actores de grupos de ransomware',
    goal=(
        'Identificar los actores más influyentes dentro del grupo Conti '
        'analizando sus roles, niveles de confianza y resúmenes de actividad.'
    ),
    backstory=(
        'Eres un investigador de CTI especializado en perfilado de actores. '
        'Has analizado centenares de leaks de grupos criminales. '
        'Tu metodología prioriza el rol funcional sobre la actividad bruta: '
        'un "leader" con pocos mensajes puede ser más relevante que un "operator" prolífico. '
        'Respondes en español con rigor analítico.'
    ),
    llm=llm,
    verbose=True,
)

# ─── AGENTE 2: ANALISTA ───────────────────────────────────────────────────────
# Su trabajo es analizar los patrones de comunicación de los actores clave
# identificados por el investigador.
analista = Agent(
    role='Analista de patrones de comunicación en grupos criminales',
    goal=(
        'Determinar qué tipos de actividad (técnica, operacional, financiera, etc.) '
        'caracterizan a cada actor clave de Conti, basándose en la distribución '
        'de categorías de sus mensajes.'
    ),
    backstory=(
        'Eres un analista especializado en OSINT y análisis de comunicaciones. '
        'Sabes que la categoría dominante en los mensajes de un actor revela su función real: '
        'muchos mensajes "technical" sugieren un developer, muchos "financial" sugieren '
        'un negotiator o manager. Respondes en español con conclusiones concretas.'
    ),
    llm=llm,
    verbose=True,
)

# ─── AGENTE 3: REDACTOR ───────────────────────────────────────────────────────
# Su trabajo es sintetizar los hallazgos de los dos agentes anteriores
# en un informe de inteligencia estructurado y accionable.
redactor = Agent(
    role='Redactor de informes de Threat Intelligence',
    goal=(
        'Producir un informe de inteligencia táctica sobre el grupo Conti '
        'que sintetice el perfilado de actores y los patrones de comunicación '
        'en conclusiones accionables para un equipo de defensa.'
    ),
    backstory=(
        'Eres un redactor técnico con experiencia en informes de threat intelligence '
        'al estilo de PRODAFT, CrowdStrike o Mandiant. '
        'Tus informes son concisos, estructurados y orientados a la toma de decisiones. '
        'Nunca incluyes especulaciones sin base en los datos. '
        'Escribes siempre en español.'
    ),
    llm=llm,
    verbose=True,
)

print('Tres agentes definidos.')

Tres agentes definidos.


## Definir las tres tareas

El parámetro `context` es la clave del trabajo encadenado: le dice a CrewAI que el output de una tarea debe estar disponible como contexto cuando se ejecute la siguiente.

In [5]:
# ─── TAREA 1: INVESTIGACIÓN DE ACTORES ────────────────────────────────────────
# El contexto completo de perfiles se incrusta directamente en la descripción.
# El agente tendrá todos los datos que necesita para razonar.
tarea_investigacion = Task(
    description=(
        f'Analiza los siguientes perfiles de actores del grupo Conti y selecciona '
        f'los 5 más relevantes para la investigación.\n\n'
        f'{TEXTO_PERFILES}\n\n'
        f'Para cada actor seleccionado, justifica por qué es relevante '
        f'(considera: rol, nivel de confianza, y qué nos dice su resumen de actividad).'
    ),
    expected_output=(
        'Una lista numerada de exactamente 5 actores con el formato:\n'
        '1. [nombre_actor] (rol: X, confianza: Y): [justificación de 1-2 frases]'
    ),
    agent=investigador,
)

# ─── TAREA 2: ANÁLISIS DE PATRONES ────────────────────────────────────────────
# Esta tarea recibe el output de tarea_investigacion como contexto adicional.
# El analista verá tanto los datos de categorías como los 5 actores identificados.
tarea_analisis = Task(
    description=(
        f'Analiza los patrones de comunicación de los actores del grupo Conti.\n\n'
        f'{TEXTO_CATEGORIAS}\n\n'
        f'Foco especial en los actores clave identificados en el análisis anterior. '
        f'Para cada uno de esos actores, determina qué revela su distribución de '
        f'categorías sobre su función real dentro del grupo.'
    ),
    expected_output=(
        'Un análisis por actor (los 5 del análisis anterior) con el formato:\n'
        '- [nombre_actor]: categoría dominante = X (N msgs). '
        'Interpretación: [qué función sugiere esta distribución]'
    ),
    agent=analista,
    context=[tarea_investigacion],  # <-- el output de tarea 1 llega aquí como contexto
)

# ─── TAREA 3: REDACCIÓN DEL INFORME ──────────────────────────────────────────
# El redactor recibe el output de AMBAS tareas anteriores como contexto.
# No necesita ver los datos crudos — solo los análisis ya elaborados.
tarea_informe = Task(
    description=(
        'Redacta un informe de Threat Intelligence sobre el grupo Conti '
        'basándote en los análisis de los agentes anteriores.\n\n'
        'El informe debe incluir:\n'
        '1. Una caracterización del grupo (2-3 frases)\n'
        '2. Los actores clave y sus roles funcionales (lista)\n'
        '3. Patrones de comunicación relevantes para equipos de defensa (párrafo)\n'
        '4. Una conclusión con la hipótesis más sólida sobre la estructura interna del grupo'
    ),
    expected_output=(
        'Un informe estructurado con las cuatro secciones indicadas. '
        'Máximo 400 palabras. Tono profesional, estilo CTI.'
    ),
    agent=redactor,
    context=[tarea_investigacion, tarea_analisis],  # <-- recibe AMBOS análisis
)

print('Tres tareas definidas con dependencias encadenadas.')

Tres tareas definidas con dependencias encadenadas.


In [6]:
# ─── CREAR Y LANZAR EL CREW ──────────────────────────────────────────────────
# Reunimos a los tres agentes y sus tareas en un Crew.
# Process.sequential garantiza que las tareas se ejecutan en orden:
# investigación → análisis → redacción.
# El contexto fluye automáticamente entre ellas gracias al parámetro `context`.

crew_conti = Crew(
    agents=[investigador, analista, redactor],
    tasks=[tarea_investigacion, tarea_analisis, tarea_informe],
    process=Process.sequential,
    verbose=True,
)

print('Lanzando el crew... (puede tardar 2-5 minutos con qwen2.5:14b)')
print('Observa cómo cada agente razona antes de producir su output.\n')

resultado = kickoff(crew_conti)

Lanzando el crew... (puede tardar 2-5 minutos con qwen2.5:14b)
Observa cómo cada agente razona antes de producir su output.



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 286366b7-96b1-4d7a-9a3e-d73ae678e94f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analiza los siguientes perfiles de actores del grupo Conti y selecciona los 5 más relevantes para la     │
│  investigación.                                                                                                 │
│                                                                                                                 │
│  PERFILES DE ACTORES DE CONTI:                                                                                  │
│  - target: rol=operator, confianza=medium. The individual appears to be an operator involved in executing       │
│  tasks and communicating with other members of the Conti ransomware group. They seem to ...                     │
│  - bentley: rol=operator, confianza=high. The individual appears to be an operator responsible for deploying    │
│  and testing ransomware, managing the deployment of loaders and ensuring that they e...                         │
│  - tl1: rol=operator, confianza=high. This individual appears to be an operator within the Conti ransomware     │
│  group, responsible for technical operations such as network reconnaissance and e...                            │
│  - stern: rol=operator, confianza=medium. The individual appears to be an operator responsible for managing     │
│  the ransomware deployment and possibly handling payments. They communicate about pay...                        │
│  - defender: rol=support, confianza=high. This individual appears to be a support member responsible for        │
│  maintaining communication channels and ensuring technical stability within the Conti ra...                     │
│  - hof: rol=developer, confianza=high. This individual appears to be a developer within the Conti ransomware    │
│  group, responsible for coding and testing malware components such as DLL files a...                            │
│  - user8: rol=operator, confianza=high. The individual is actively involved in the technical operations of      │
│  network exploitation, including attempting logins and pinging trust domains. They a...                         │
│  - mango: rol=unknown, confianza=low. Expecting value: line 1 column 1 (char 0)...                              │
│  - driver: rol=operator, confianza=medium. The individual appears to be an operator responsible for managing    │
│  the encryption and distribution of ransomware packages. They communicate about prior...                        │
│  - deploy: rol=operator, confianza=high. This individual appears to be an operator responsible for deploying    │
│  and managing ransomware components, such as DLLs and EXEs. They communicate about ...                          │
│  - mushroom: rol=developer, confianza=high. The individual is responsible for developing and maintaining the    │
│  ransomware's codebase, including its loader and communication protocols. They also pr...                       │
│  - bio: rol=operator, confianza=medium. This individual appears to be an operator within the Conti ransomware   │
│  group, responsible for handling day-to-day operations such as responding to vict...                            │
│  - baget: rol=operator, confianza=high. The individual appears to be an operator within the Conti ransomware    │
│  group, responsible for executing technical tasks such as deploying and managing m...                           │
│  - ttrr: rol=developer, confianza=high. ttrr appears to be a developer working on technical aspects of the      │
│  Conti ransomware infrastructure, dealing with issues such as library changes and HT...                         │
│  - veron: rol=support, confianza=medium. The individual

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de actores de grupos de ransomware                                                         │
│                                                                                                                 │
│  Task: Analiza los siguientes perfiles de actores del grupo Conti y selecciona los 5 más relevantes para la     │
│  investigación.                                                                                                 │
│                                                                                                                 │
│  PERFILES DE ACTORES DE CONTI:                                                                                  │
│  - target: rol=operator, confianza=medium. The individual appears to be an operator involved in executing       │
│  tasks and communicating with other members of the Conti ransomware group. They seem to ...                     │
│  - bentley: rol=operator, confianza=high. The individual appears to be an operator responsible for deploying    │
│  and testing ransomware, managing the deployment of loaders and ensuring that they e...                         │
│  - tl1: rol=operator, confianza=high. This individual appears to be an operator within the Conti ransomware     │
│  group, responsible for technical operations such as network reconnaissance and e...                            │
│  - stern: rol=operator, confianza=medium. The individual appears to be an operator responsible for managing     │
│  the ransomware deployment and possibly handling payments. They communicate about pay...                        │
│  - defender: rol=support, confianza=high. This individual appears to be a support member responsible for        │
│  maintaining communication channels and ensuring technical stability within the Conti ra...                     │
│  - hof: rol=developer, confianza=high. This individual appears to be a developer within the Conti ransomware    │
│  group, responsible for coding and testing malware components such as DLL files a...                            │
│  - user8: rol=operator, confianza=high. The individual is actively involved in the technical operations of      │
│  network exploitation, including attempting logins and pinging trust domains. They a...                         │
│  - mango: rol=unknown, confianza=low. Expecting value: line 1 column 1 (char 0)...                              │
│  - driver: rol=operator, confianza=medium. The individual appears to be an operator responsible for managing    │
│  the encryption and distribution of ransomware packages. They communicate about prior...                        │
│  - deploy: rol=operator, confianza=high. This individual appears to be an operator responsible for deploying    │
│  and managing ransomware components, such as DLLs and EXEs. They communicate about ...                          │
│  - mushroom: rol=developer, confianza=high. The individual is responsible for developing and maintaining the    │
│  ransomware's codebase, including its loader and communication protocols. They also pr...                       │
│  - bio: rol=operator, confianza=medium. This individual appears to be an operator within the Conti ransomware   │
│  group, responsible for handling day-to-day operations such as responding to vict...                            │
│  - baget: rol=operator, confianza=high. The individual appears to be an operator within the Conti ransomware    │
│  group, responsible for executing technical tasks such as deploying and managing m...                           │
│  - ttrr: rol=developer, confianza=high. ttrr appears to be a developer working on technical aspects of the      │
│  Conti ransomware infrastructure, dealing with issues s

[Finalize] todos_count=0, todos_with_results=0


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor de informes de Threat Intelligence                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Informe de Threat Intelligence sobre el Grupo Conti                                                        │
│                                                                                                                 │
│  #### Caracterización del Grupo (2-3 frases)                                                                    │
│  El grupo Conti se caracteriza por ser una organización cibercriminal altamente sofisticada y jerárquica,       │
│  conocida por su uso intensivo de ransomware y sus ataques en gran escala a instituciones críticas. Los         │
│  miembros clave se enfocan principalmente en el desarrollo técnico y la operación del malware dentro de las     │
│  redes comprometidas.                                                                                           │
│                                                                                                                 │
│  #### Actores Clave y Roles Funcionales                                                                         │
│  1. **hof**: Desarrollador (Confianza: alta)                                                                    │
│     - Responsable del diseño y pruebas de los componentes principales del ransomware.                           │
│  2. **mushroom**: Desarrollador (Confianza: alta)                                                               │
│     - Mantenimiento del núcleo del ransomware incluyendo módulos y protocolos críticos.                         │
│  3. **ttrr**: Desarrollador (Confianza: alta)                                                                   │
│     - Solución de problemas complejos relacionados con la infraestructura técnica del malware.                  │
│  4. **user9**: Operador (Confianza: mediana)                                                                    │
│     - Movimiento lateral dentro de redes comprometidas para ampliar las actividades y maximizar ganancias.      │
│  5. **price**: Operador (Confianza: mediana)                                                                    │
│     - Identificación y explotación de vulnerabilidades tecnológicas importantes.                                │
│                                                                                                                 │
│  #### Patrones de Comunicación Relevantes para Equipos de Defensa                                               │
│  El análisis muestra que los actores clave concentran sus comunicaciones técnicas (más del 50% de las           │
│  interacciones) relacionadas con el desarrollo y mantenimiento del ransomware Conti. Estas conversaciones       │
│  indican una organización altamente estructurada donde:                                                         │
│  - Los desarrolladores (hof, mushroom, ttrr) se centran en mejoras constantes y pruebas de los módulos          │
│  críticos del malware.                                                                                          │
│  - Los operadores (user9, price) se comunican principalmente sobre estrategias técnicas para expandir el        │
│  alcance del ransomware dentro del entorno comprometido.                                                        │
│                                                                                                                 │
│  Estos patrones de comunicación evidencian un trabajo e

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un informe de Threat Intelligence sobre el grupo Conti basándote en los análisis de los agentes  │
│  anteriores.                                                                                                    │
│                                                                                                                 │
│  El informe debe incluir:                                                                                       │
│  1. Una caracterización del grupo (2-3 frases)                                                                  │
│  2. Los actores clave y sus roles funcionales (lista)                                                           │
│  3. Patrones de comunicación relevantes para equipos de defensa (párrafo)                                       │
│  4. Una conclusión con la hipótesis más sólida sobre la estructura interna del grupo                            │
│  Agent: Redactor de informes de Threat Intelligence                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 286366b7-96b1-4d7a-9a3e-d73ae678e94f                                                                       │
│  Final Output: ### Informe de Threat Intelligence sobre el Grupo Conti                                          │
│                                                                                                                 │
│  #### Caracterización del Grupo (2-3 frases)                                                                    │
│  El grupo Conti se caracteriza por ser una organización cibercriminal altamente sofisticada y jerárquica,       │
│  conocida por su uso intensivo de ransomware y sus ataques en gran escala a instituciones críticas. Los         │
│  miembros clave se enfocan principalmente en el desarrollo técnico y la operación del malware dentro de las     │
│  redes comprometidas.                                                                                           │
│                                                                                                                 │
│  #### Actores Clave y Roles Funcionales                                                                         │
│  1. **hof**: Desarrollador (Confianza: alta)                                                                    │
│     - Responsable del diseño y pruebas de los componentes principales del ransomware.                           │
│  2. **mushroom**: Desarrollador (Confianza: alta)                                                               │
│     - Mantenimiento del núcleo del ransomware incluyendo módulos y protocolos críticos.                         │
│  3. **ttrr**: Desarrollador (Confianza: alta)                                                                   │
│     - Solución de problemas complejos relacionados con la infraestructura técnica del malware.                  │
│  4. **user9**: Operador (Confianza: mediana)                                                                    │
│     - Movimiento lateral dentro de redes comprometidas para ampliar las actividades y maximizar ganancias.      │
│  5. **price**: Operador (Confianza: mediana)                                                                    │
│     - Identificación y explotación de vulnerabilidades tecnológicas importantes.                                │
│                                                                                                                 │
│  #### Patrones de Comunicación Relevantes para Equipos de Defensa                                               │
│  El análisis muestra que los actores clave concentran sus comunicaciones técnicas (más del 50% de las           │
│  interacciones) relacionadas con el desarrollo y mantenimiento del ransomware Conti. Estas conversaciones       │
│  indican una organización altamente estructurada donde:                                                         │
│  - Los desarrolladores (hof, mushroom, ttrr) se centran en mejoras constantes y pruebas de los módulos          │
│  críticos del malware.                                                                                          │
│  - Los operadores (user9, price) se comunican principalmente sobre estrategias técnicas para expandir el        │
│  alcance del ransomware dentro del entorno comprometido.                                                        │
│                                                                                                                 │
│  Estos patrones de comunicación evidencian un trabajo 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
# ─── RESULTADO FINAL ─────────────────────────────────────────────────────────
# El resultado del Crew es el output de la ÚLTIMA tarea (el informe del redactor).
# Los outputs intermedios (investigador, analista) sirvieron como contexto
# pero no se devuelven directamente — fluyen internamente.

print('=' * 70)
print('INFORME DE THREAT INTELLIGENCE — GRUPO CONTI')
print('Generado por crew de 3 agentes locales (qwen2.5:14b + Ollama)')
print('=' * 70)
print(resultado.raw)

INFORME DE THREAT INTELLIGENCE — GRUPO CONTI
Generado por crew de 3 agentes locales (qwen2.5:14b + Ollama)
### Informe de Threat Intelligence sobre el Grupo Conti

#### Caracterización del Grupo (2-3 frases)
El grupo Conti se caracteriza por ser una organización cibercriminal altamente sofisticada y jerárquica, conocida por su uso intensivo de ransomware y sus ataques en gran escala a instituciones críticas. Los miembros clave se enfocan principalmente en el desarrollo técnico y la operación del malware dentro de las redes comprometidas.

#### Actores Clave y Roles Funcionales
1. **hof**: Desarrollador (Confianza: alta)
   - Responsable del diseño y pruebas de los componentes principales del ransomware.
2. **mushroom**: Desarrollador (Confianza: alta) 
   - Mantenimiento del núcleo del ransomware incluyendo módulos y protocolos críticos.
3. **ttrr**: Desarrollador (Confianza: alta)
   - Solución de problemas complejos relacionados con la infraestructura técnica del malware.
4. **user9*

## Discusión: ¿qué haría falta para un sistema de producción?

Lo que acabamos de construir es un prototipo funcional. Para convertirlo en un sistema real de CTI faltan varias cosas:

### 1. Evaluación
Ahora el output es texto libre. No sabemos si el informe es bueno o malo salvo leyéndolo. Un sistema productivo necesita:
- Una función de scoring automático (otro LLM que valúe el output)
- Benchmarks con informes de referencia (PRODAFT, CrowdStrike)

### 2. Memoria persistente
Nuestra crew empieza desde cero cada vez. No recuerda lo que analizó ayer. CrewAI soporta memoria persistente con una base de datos vectorial — lo que ya sabemos hacer desde bloque4.

### 3. Human-in-the-loop
Algunos sistemas de CTI requieren que un analista humano apruebe los hallazgos antes de que el redactor los incluya en el informe. CrewAI soporta `human_input=True` en las tareas críticas.

### 4. Modelos más capaces para razonamiento complejo
`qwen2.5:14b` funciona bien para tareas de resumen y clasificación, pero un análisis de amenazas real requiere modelos más grandes (32B+) o modelos especializados en seguridad.

---

**Siguiente**: en el notebook `02_agentes_con_herramientas.ipynb` añadimos **herramientas** al agente — funciones Python que el LLM puede invocar para buscar en los datos de forma precisa.